In [0]:
storage_account = "dnvenergystorage"
container = "building-data"
account_key = "uR0lF4HmJo/BqqH/EDo5qzgzkoGduiiGHTL94B3ZrTWvsx/l20Us9oaX/4BxtKPCeXVF4OMkswna+AStxmynZQ=="

# Method 1: Hadoop config (more reliable)
spark._jsc.hadoopConfiguration().set(
    f"fs.azure.account.key.{storage_account}.blob.core.windows.net",
    account_key
)

# Test connection
base_path = f"wasbs://{container}@{storage_account}.blob.core.windows.net"
print(f"Testing: {base_path}")

try:
    files = dbutils.fs.ls(base_path)
    print("✅ Connected! Files:")
    for f in files:
        print(f"  - {f.name}")
except Exception as e:
    print(f"❌ Still failed: {e}")
    
    # Try alternative protocol
    print("\nTrying alternative protocol (abfss)...")
    alt_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net"
    try:
        files = dbutils.fs.ls(alt_path)
        print("✅ Alternative worked! Files:")
        for f in files:
            print(f"  - {f.name}")
    except Exception as e2:
        print(f"❌ Alternative also failed: {e2}")

Testing: wasbs://building-data@dnvenergystorage.blob.core.windows.net
✅ Connected! Files:
  - chilledwater_cleaned.csv
  - electricity_cleaned.csv
  - hotwater_cleaned.csv
  - metadata.csv
  - weather.csv


In [0]:
# CELL 2: Load Electricity Data (NO CHANGES)
from pyspark.sql import functions as F

print("Loading electricity data...")
df_elec = spark.read.csv(f"{base_path}/electricity_cleaned.csv", header=True, inferSchema=True)
df_meta = spark.read.csv(f"{base_path}/metadata.csv", header=True, inferSchema=True)
df_weather = spark.read.csv(f"{base_path}/weather.csv", header=True, inferSchema=True)

print(f"✅ Electricity: {df_elec.count()} rows, {len(df_elec.columns)} columns")

# Get numeric building columns
all_cols = df_elec.columns
building_cols = [c for c in all_cols if c != 'timestamp']

numeric_cols = []
for col_name in building_cols:
    col_type = dict(df_elec.dtypes)[col_name]
    if col_type in ['double', 'float', 'int', 'bigint']:
        numeric_cols.append(col_name)

print(f"✅ Found {len(numeric_cols)} numeric building columns")

# USE ALL BUILDINGS
sampled_buildings = numeric_cols

# Filter to 2017
df_elec_sample = df_elec.select(['timestamp'] + sampled_buildings).filter(
    (F.col('timestamp') >= '2017-01-01') & (F.col('timestamp') < '2017-07-01')
)

# Convert to long format
stack_expr = "stack({}, {}) as (building_id, meter_reading)".format(
    len(sampled_buildings),
    ', '.join([f"'{b}', `{b}`" for b in sampled_buildings])
)

df_long = df_elec_sample.selectExpr('timestamp', stack_expr).filter(F.col('meter_reading').isNotNull())

print(f"\n✅ Long format: {df_long.count():,} readings")
print(f"✅ Buildings: {df_long.select('building_id').distinct().count()}")

df_long.show(5)

In [0]:
# CELL 3: Load HVAC Data (FIXED - filter non-numeric columns)
print("Loading multi-meter data...")

# Load chilled water (cooling) - FILTER TO NUMERIC ONLY
try:
    df_cooling_raw = spark.read.csv(f"{base_path}/chilledwater_cleaned.csv", header=True, inferSchema=True)
    
    # Get ONLY numeric columns (same as electricity)
    cooling_all_cols = [c for c in df_cooling_raw.columns if c != 'timestamp']
    cooling_numeric_cols = []
    
    for col_name in cooling_all_cols:
        col_type = dict(df_cooling_raw.dtypes)[col_name]
        if col_type in ['double', 'float', 'int', 'bigint']:
            cooling_numeric_cols.append(col_name)
    
    print(f"Cooling: {len(cooling_numeric_cols)} numeric columns (filtered from {len(cooling_all_cols)} total)")
    
    # Only use buildings that exist in our sample
    cooling_cols = [c for c in cooling_numeric_cols if c in sampled_buildings]
    
    if len(cooling_cols) > 0:
        df_cooling_sample = df_cooling_raw.select(['timestamp'] + cooling_cols).filter(
            (F.col('timestamp') >= '2017-01-01') & (F.col('timestamp') < '2017-07-01')
        )
        
        # Convert to long format
        stack_expr_cooling = "stack({}, {}) as (building_id, cooling_reading)".format(
            len(cooling_cols),
            ', '.join([f"'{b}', `{b}`" for b in cooling_cols])
        )
        
        df_cooling = df_cooling_sample.selectExpr('timestamp', stack_expr_cooling).filter(
            F.col('cooling_reading').isNotNull()
        )
        
        print(f"✅ Cooling: {df_cooling.select('building_id').distinct().count()} buildings")
    else:
        print("⚠️  No cooling buildings in sample")
        df_cooling = None
        
except Exception as e:
    print(f"⚠️  No cooling data: {e}")
    df_cooling = None

# Load hot water (heating) - FILTER TO NUMERIC ONLY
try:
    df_heating_raw = spark.read.csv(f"{base_path}/hotwater_cleaned.csv", header=True, inferSchema=True)
    
    # Get ONLY numeric columns
    heating_all_cols = [c for c in df_heating_raw.columns if c != 'timestamp']
    heating_numeric_cols = []
    
    for col_name in heating_all_cols:
        col_type = dict(df_heating_raw.dtypes)[col_name]
        if col_type in ['double', 'float', 'int', 'bigint']:
            heating_numeric_cols.append(col_name)
    
    print(f"Heating: {len(heating_numeric_cols)} numeric columns (filtered from {len(heating_all_cols)} total)")
    
    heating_cols = [c for c in heating_numeric_cols if c in sampled_buildings]
    
    if len(heating_cols) > 0:
        df_heating_sample = df_heating_raw.select(['timestamp'] + heating_cols).filter(
            (F.col('timestamp') >= '2017-01-01') & (F.col('timestamp') < '2017-07-01')
        )
        
        stack_expr_heating = "stack({}, {}) as (building_id, heating_reading)".format(
            len(heating_cols),
            ', '.join([f"'{b}', `{b}`" for b in heating_cols])
        )
        
        df_heating = df_heating_sample.selectExpr('timestamp', stack_expr_heating).filter(
            F.col('heating_reading').isNotNull()
        )
        
        print(f"✅ Heating: {df_heating.select('building_id').distinct().count()} buildings")
    else:
        print("⚠️  No heating buildings in sample")
        df_heating = None
        
except Exception as e:
    print(f"⚠️  No heating data: {e}")
    df_heating = None

print("\n✅ Multi-meter data loading complete")


FEATURE ENGINEERING

In [0]:
# CELL 1: Temporal and Operational Features (ENHANCED)
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("Engineering ENHANCED features...\n")

# Temporal features (expanded)
df_long = df_long.withColumn('hour', F.hour('timestamp')) \
                 .withColumn('day_of_week', F.dayofweek('timestamp')) \
                 .withColumn('is_weekend', F.when(F.col('day_of_week').isin([1,7]), 1).otherwise(0)) \
                 .withColumn('is_weekday', F.when(~F.col('day_of_week').isin([1,7]), 1).otherwise(0)) \
                 .withColumn('is_business_hours', 
                             F.when((F.col('hour') >= 8) & (F.col('hour') <= 18) & (F.col('day_of_week').isin([2,3,4,5,6])), 1).otherwise(0)) \
                 .withColumn('is_night', 
                             F.when((F.col('hour') >= 23) | (F.col('hour') <= 5), 1).otherwise(0))

# Building attributes
df_long = df_long.withColumn('building_type', F.split(F.col('building_id'), '_').getItem(1)) \
                 .withColumn('site', F.split(F.col('building_id'), '_').getItem(0))

# Windows
window_7d = Window.partitionBy('building_id').orderBy('timestamp').rowsBetween(-168, 0)
window_24h = Window.partitionBy('building_id').orderBy('timestamp').rowsBetween(-24, 0)

# Rolling features (enhanced)
print("Calculating rolling features...")
df_long = df_long.withColumn('baseload_7day', 
                              F.expr('percentile_approx(meter_reading, 0.1)').over(window_7d))

df_long = df_long.withColumn('rolling_avg_24h', F.avg('meter_reading').over(window_24h))

df_long = df_long.withColumn('rolling_std_24h', F.stddev('meter_reading').over(window_24h))

# Derived metrics
df_long = df_long.withColumn('peak_ratio', F.col('meter_reading') / (F.col('baseload_7day') + 0.01))

df_long = df_long.withColumn('volatility', F.col('rolling_std_24h') / (F.col('rolling_avg_24h') + 0.01))

print("✅ Temporal features created")

Engineering ENHANCED features...

Calculating rolling features...
✅ Temporal features created


In [0]:
# CELL 2: Weather Integration (SAME)
print("Merging weather data...")

df_weather_clean = df_weather.select(
    F.col('timestamp').alias('weather_timestamp'),
    F.col('site_id').alias('weather_site'),
    F.col('airTemperature').alias('temperature'),
    F.col('dewTemperature').alias('dew_temperature')
).dropna()

df_long = df_long.join(
    F.broadcast(df_weather_clean),
    (df_long['timestamp'] == df_weather_clean['weather_timestamp']) & 
    (df_long['site'] == df_weather_clean['weather_site']),
    'left'
).select(
    df_long['building_id'],
    df_long['timestamp'],
    df_long['hour'],
    df_long['day_of_week'],
    df_long['meter_reading'],
    df_long['building_type'],
    df_long['is_weekend'],
    df_long['is_weekday'],
    df_long['is_business_hours'],
    df_long['is_night'],
    df_long['baseload_7day'],
    df_long['peak_ratio'],
    df_long['rolling_avg_24h'],
    df_long['rolling_std_24h'],
    df_long['volatility'],
    df_weather_clean['temperature'].alias('air_temperature'),
    df_weather_clean['dew_temperature']
)

print("✅ Weather merged")

Merging weather data...
✅ Weather merged


In [0]:
# CELL 3: Daily Aggregation (ENHANCED)
print("Aggregating to daily with rich metrics...\n")

df_daily = df_long.groupBy('building_id', F.to_date('timestamp').alias('date')).agg(
    F.mean('meter_reading').alias('avg_consumption'),
    F.max('meter_reading').alias('peak_consumption'),
    F.min('meter_reading').alias('min_consumption'),
    F.stddev('meter_reading').alias('consumption_std'),
    F.mean('baseload_7day').alias('baseload'),
    F.mean('peak_ratio').alias('avg_peak_ratio'),
    F.mean('volatility').alias('avg_volatility'),
    F.mean('air_temperature').alias('avg_temp'),
    F.max('air_temperature').alias('max_temp'),
    F.min('air_temperature').alias('min_temp'),
    F.first('building_type').alias('building_type'),
    F.max('is_weekend').alias('is_weekend'),
    F.sum('is_business_hours').alias('business_hours_count'),
    F.sum('is_night').alias('night_hours_count')
).dropna()

# Temperature range
df_daily = df_daily.withColumn('temp_range', F.col('max_temp') - F.col('min_temp'))

print(f"✅ Daily features: {df_daily.count()} rows")
print(f"✅ Buildings: {df_daily.select('building_id').distinct().count()}")

df_daily.show(10)

Aggregating to daily with rich metrics...

✅ Daily features: 265109 rows
✅ Buildings: 1533
+--------------------+----------+------------------+----------------+---------------+------------------+------------------+------------------+-------------------+------------------+--------+--------+-------------+----------+--------------------+-----------------+------------------+
|         building_id|      date|   avg_consumption|peak_consumption|min_consumption|   consumption_std|          baseload|    avg_peak_ratio|     avg_volatility|          avg_temp|max_temp|min_temp|building_type|is_weekend|business_hours_count|night_hours_count|        temp_range|
+--------------------+----------+------------------+----------------+---------------+------------------+------------------+------------------+-------------------+------------------+--------+--------+-------------+----------+--------------------+-----------------+------------------+
|Bear_assembly_Bea...|2017-01-01|27.072916666666668|        

In [0]:
# CELL 4: Multi-Meter Integration (NEW)
print("Integrating HVAC meters...\n")

# Add cooling data if available
if df_cooling is not None:
    df_cooling_daily = df_cooling.groupBy('building_id', F.to_date('timestamp').alias('date')).agg(
        F.mean('cooling_reading').alias('cooling_avg'),
        F.max('cooling_reading').alias('cooling_peak')
    )
    
    df_daily = df_daily.join(df_cooling_daily, on=['building_id', 'date'], how='left')
    print(f"✅ Cooling data merged: {df_daily.filter(F.col('cooling_avg').isNotNull()).count()} building-days")

# Add heating data if available
if df_heating is not None:
    df_heating_daily = df_heating.groupBy('building_id', F.to_date('timestamp').alias('date')).agg(
        F.mean('heating_reading').alias('heating_avg'),
        F.max('heating_reading').alias('heating_peak')
    )
    
    df_daily = df_daily.join(df_heating_daily, on=['building_id', 'date'], how='left')
    print(f"✅ Heating data merged: {df_daily.filter(F.col('heating_avg').isNotNull()).count()} building-days")

# Calculate multi-meter features
df_daily = df_daily.withColumn(
    'total_energy',
    F.col('avg_consumption') + 
    F.coalesce(F.col('heating_avg'), F.lit(0)) + 
    F.coalesce(F.col('cooling_avg'), F.lit(0))
)

# HVAC ratio
if df_cooling is not None and df_heating is not None:
    df_daily = df_daily.withColumn(
        'hvac_ratio',
        (F.coalesce(F.col('heating_avg'), F.lit(0)) + F.coalesce(F.col('cooling_avg'), F.lit(0))) / 
        (F.col('avg_consumption') + 0.01)
    )
    print("✅ HVAC ratio calculated")

# Cache for modeling
df_daily.cache()
print(f"\n✅ Feature engineering complete: {len(df_daily.columns)} columns")

Integrating HVAC meters...

✅ Cooling data merged: 90500 building-days
✅ Heating data merged: 30380 building-days
✅ HVAC ratio calculated

✅ Feature engineering complete: 23 columns


In [0]:
# CELL 5: Building-Level Behavioral Features (NEW)
print("Calculating behavioral patterns...\n")

# Separate weekend/weekday consumption
df_weekday = df_daily.filter(F.col('is_weekend') == 0).groupBy('building_id').agg(
    F.mean('avg_consumption').alias('weekday_avg')
)

df_weekend = df_daily.filter(F.col('is_weekend') == 1).groupBy('building_id').agg(
    F.mean('avg_consumption').alias('weekend_avg')
)

# Night vs day consumption (from hourly data)
df_night = df_long.filter(F.col('is_night') == 1).groupBy('building_id').agg(
    F.mean('meter_reading').alias('night_avg')
)

df_day = df_long.filter(F.col('is_night') == 0).groupBy('building_id').agg(
    F.mean('meter_reading').alias('day_avg')
)

# Temperature sensitivity (correlation - approximate using covariance)
from pyspark.sql.functions import corr

df_temp_sens = df_daily.groupBy('building_id').agg(
    corr('avg_temp', 'avg_consumption').alias('temp_sensitivity')
)

# Consumption CV (coefficient of variation)
df_cv = df_daily.groupBy('building_id').agg(
    (F.stddev('avg_consumption') / F.mean('avg_consumption')).alias('consumption_cv')
)

print("✅ Behavioral patterns calculated")

Calculating behavioral patterns...

✅ Behavioral patterns calculated


MLFLOW TRAINING

In [0]:
# CELL 1: Prepare Building-Level Data with Rich Features (ENHANCED)
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import xgboost as xgb
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import mean_absolute_error, silhouette_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
import pandas as pd
import numpy as np

mlflow.set_experiment("/Users/danielgonzalezmuela@gmail.com/dnv-building-energy-ml-enhanced")

print("Preparing ENHANCED building-level data...\n")

# Aggregate to building level
df_bldg_spark = df_daily.groupBy('building_id').agg(
    F.mean('avg_consumption').alias('avg_consumption'),
    F.mean('peak_consumption').alias('peak_consumption'),
    F.mean('min_consumption').alias('min_consumption'),
    F.mean('consumption_std').alias('consumption_std'),
    F.mean('baseload').alias('baseload'),
    F.mean('avg_peak_ratio').alias('avg_peak_ratio'),
    F.mean('avg_volatility').alias('avg_volatility'),
    F.mean('avg_temp').alias('avg_temp'),
    F.mean('temp_range').alias('temp_range'),
    F.first('building_type').alias('building_type'),
    F.mean('total_energy').alias('total_energy')
)

# Convert to pandas
df_bldg = df_bldg_spark.toPandas()

# Add behavioral features
df_bldg = df_bldg.merge(df_weekday.toPandas(), on='building_id', how='left')
df_bldg = df_bldg.merge(df_weekend.toPandas(), on='building_id', how='left')
df_bldg = df_bldg.merge(df_night.toPandas(), on='building_id', how='left')
df_bldg = df_bldg.merge(df_day.toPandas(), on='building_id', how='left')
df_bldg = df_bldg.merge(df_temp_sens.toPandas(), on='building_id', how='left')
df_bldg = df_bldg.merge(df_cv.toPandas(), on='building_id', how='left')

# Calculate ratios
df_bldg['weekend_ratio'] = df_bldg['weekend_avg'] / df_bldg['weekday_avg']
df_bldg['night_ratio'] = df_bldg['night_avg'] / df_bldg['day_avg']

# Add HVAC if available
if 'cooling_avg' in df_daily.columns:
    df_hvac_cool = df_daily.groupBy('building_id').agg(F.mean('cooling_avg').alias('cooling_avg')).toPandas()
    df_bldg = df_bldg.merge(df_hvac_cool, on='building_id', how='left')

if 'heating_avg' in df_daily.columns:
    df_hvac_heat = df_daily.groupBy('building_id').agg(F.mean('heating_avg').alias('heating_avg')).toPandas()
    df_bldg = df_bldg.merge(df_hvac_heat, on='building_id', how='left')

if 'hvac_ratio' in df_daily.columns:
    df_hvac_ratio = df_daily.groupBy('building_id').agg(F.mean('hvac_ratio').alias('hvac_ratio')).toPandas()
    df_bldg = df_bldg.merge(df_hvac_ratio, on='building_id', how='left')

print(f"Buildings: {len(df_bldg)}")
print(f"Features: {df_bldg.shape[1]}\n")

2026/01/02 09:36:55 INFO mlflow.tracking.fluent: Experiment with name '/Users/danielgonzalezmuela@gmail.com/dnv-building-energy-ml-enhanced' does not exist. Creating a new experiment.


Preparing ENHANCED building-level data...

Buildings: 1533
Features: 23



In [0]:
# CELL 2: MODEL 1 - Temporal Clustering (ENHANCED - but keep simple on Azure)
print("="*70)
print("MODEL 1: K-MEANS CLUSTERING (Operational Profiles)")
print("="*70)

# Use rich feature set for clustering
cluster_features = ['baseload', 'avg_peak_ratio', 'avg_consumption', 
                    'weekend_ratio', 'night_ratio', 'consumption_cv']

X_cluster = df_bldg[cluster_features].fillna(0)

# Standardize
scaler = StandardScaler()
X_cluster_scaled = scaler.fit_transform(X_cluster)

# Find optimal k (test 3-6)
k_range = range(3, 7)
silhouettes = []

for k in k_range:
    kmeans_test = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_test = kmeans_test.fit_predict(X_cluster_scaled)
    silhouettes.append(silhouette_score(X_cluster_scaled, labels_test))
    print(f"k={k}: Silhouette={silhouettes[-1]:.3f}")

optimal_k = k_range[np.argmax(silhouettes)]
print(f"\nOptimal k: {optimal_k}")

# Train with optimal k
with mlflow.start_run(run_name="clustering_enhanced"):
    kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
    df_bldg['cluster'] = kmeans.fit_predict(X_cluster_scaled)
    
    sil_score = silhouette_score(X_cluster_scaled, df_bldg['cluster'])
    
    mlflow.log_param("n_clusters", optimal_k)
    mlflow.log_param("features_used", len(cluster_features))
    mlflow.log_param("feature_list", ', '.join(cluster_features))
    mlflow.log_metric("silhouette_score", sil_score)
    mlflow.sklearn.log_model(kmeans, "kmeans_model")
    mlflow.sklearn.log_model(scaler, "scaler")
    
    print(f"\n✅ Silhouette Score: {sil_score:.3f}")
    print(f"Cluster sizes: {df_bldg['cluster'].value_counts().sort_index().to_dict()}")

In [0]:
# CELL 3: MODEL 2 - Multi-Variate Anomaly Detection (ENHANCED)
print("\n" + "="*70)
print("MODEL 2: ANOMALY DETECTION (Multi-Variate)")
print("="*70)

# Rich feature set
anomaly_features = [
    'baseload', 'avg_peak_ratio', 'avg_consumption',
    'weekend_ratio', 'night_ratio', 'consumption_cv',
    'temp_sensitivity', 'avg_volatility'
]

# Add HVAC if available
if 'heating_avg' in df_bldg.columns:
    anomaly_features.append('heating_avg')
if 'cooling_avg' in df_bldg.columns:
    anomaly_features.append('cooling_avg')
if 'hvac_ratio' in df_bldg.columns:
    anomaly_features.append('hvac_ratio')

X_anomaly = df_bldg[anomaly_features].fillna(0)

print(f"Using {len(anomaly_features)} features:")
print(f"  {', '.join(anomaly_features)}\n")

with mlflow.start_run(run_name="anomaly_detection_enhanced"):
    iso = IsolationForest(contamination=0.15, random_state=42, n_estimators=100)
    df_bldg['is_anomaly'] = iso.fit_predict(X_anomaly)
    df_bldg['is_anomaly'] = (df_bldg['is_anomaly'] == -1).astype(int)
    
    n_anomalies = df_bldg['is_anomaly'].sum()
    
    mlflow.log_param("contamination", 0.15)
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("features_used", len(anomaly_features))
    mlflow.log_param("feature_list", ', '.join(anomaly_features))
    mlflow.log_metric("n_anomalies", n_anomalies)
    mlflow.log_metric("pct_anomalies", n_anomalies/len(df_bldg))
    mlflow.sklearn.log_model(iso, "isolation_forest_model")
    
    print(f"✅ Anomalies detected: {n_anomalies} ({n_anomalies/len(df_bldg)*100:.1f}%)")


MODEL 2: ANOMALY DETECTION (Multi-Variate)
Using 11 features:
  baseload, avg_peak_ratio, avg_consumption, weekend_ratio, night_ratio, consumption_cv, temp_sensitivity, avg_volatility, heating_avg, cooling_avg, hvac_ratio



/databricks/python/lib/python3.10/site-packages/sklearn/base.py:450: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(
2026/01/02 09:54:46 WARNING mlflow.models.model: Model logged without a signature. Signatures will be required for upcoming model registry features as they validate model inputs and denote the expected schema of model outputs. Please visit https://www.mlflow.org/docs/2.9.2/models.html#set-signature-on-logged-model for instructions on setting a model signature on your logged model.


Uploading artifacts:   0%|          | 0/5 [00:00<?, ?it/s]

✅ Anomalies detected: 230 (15.0%)


In [0]:
# CELL 4: MODEL 3 - Weekend Efficiency Prediction (ENHANCED)
print("\n" + "="*70)
print("MODEL 3: WEEKEND EFFICIENCY PREDICTION (XGBoost)")
print("="*70)

# Remove buildings with missing weekend ratio
df_model = df_bldg.dropna(subset=['weekend_ratio']).copy()

print(f"Weekend ratio statistics:")
print(f"  • Mean: {df_model['weekend_ratio'].mean():.2f}")
print(f"  • Median: {df_model['weekend_ratio'].median():.2f}")
print(f"  • Poor shutdown (>0.8): {(df_model['weekend_ratio'] > 0.8).sum()}\n")

# Encode building type
le = LabelEncoder()
df_model['building_type_encoded'] = le.fit_transform(df_model['building_type'])

# Rich feature set
xgb_features = [
    'avg_consumption', 'baseload', 'avg_peak_ratio',
    'night_ratio', 'consumption_cv', 'avg_volatility',
    'temp_sensitivity', 'building_type_encoded'
]

# Add HVAC if available
if 'heating_avg' in df_model.columns:
    xgb_features.append('heating_avg')
if 'cooling_avg' in df_model.columns:
    xgb_features.append('cooling_avg')

print(f"Using {len(xgb_features)} features:")
print(f"  {', '.join(xgb_features)}\n")

X_train = df_model[xgb_features].fillna(0)
y_train = df_model['weekend_ratio']

with mlflow.start_run(run_name="xgboost_weekend_efficiency") as run:
    model = xgb.XGBRegressor(
        n_estimators=150,
        max_depth=5,
        learning_rate=0.05,
        random_state=42,
        subsample=0.8,
        colsample_bytree=0.8
    )
    
    model.fit(X_train, y_train)
    
    predictions = model.predict(X_train)
    df_model['predicted_weekend_ratio'] = predictions
    df_model['weekend_gap'] = df_model['weekend_ratio'] - predictions
    
    mae = mean_absolute_error(y_train, predictions)
    r2 = model.score(X_train, y_train)
    
    # Flag underperformers
    df_model['poor_weekend_shutdown'] = (df_model['weekend_gap'] > 0.10).astype(int)
    n_poor = df_model['poor_weekend_shutdown'].sum()
    
    signature = infer_signature(X_train, predictions)
    
    mlflow.log_param("n_estimators", 150)
    mlflow.log_param("max_depth", 5)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("features_used", len(xgb_features))
    mlflow.log_param("feature_list", ', '.join(xgb_features))
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2_score", r2)
    mlflow.log_metric("n_poor_shutdown", n_poor)
    mlflow.log_metric("pct_poor_shutdown", n_poor/len(df_model))
    
    mlflow.sklearn.log_model(model, "xgboost_model", signature=signature)
    
    production_run_id = run.info.run_id
    
    print(f"✅ MAE: {mae:.3f}")
    print(f"✅ R²: {r2:.3f}")
    print(f"✅ Poor shutdown: {n_poor} buildings ({n_poor/len(df_model)*100:.1f}%)")

# Merge back to main dataframe
df_bldg = df_bldg.merge(
    df_model[['building_id', 'predicted_weekend_ratio', 'weekend_gap', 'poor_weekend_shutdown']],
    on='building_id',
    how='left'
)

print("\n" + "="*70)
print("✅ ALL 3 ENHANCED MODELS TRAINED & LOGGED TO MLFLOW")
print("="*70)
print(f"\nRun ID for registration: {production_run_id}")


MODEL 3: WEEKEND EFFICIENCY PREDICTION (XGBoost)
Weekend ratio statistics:
  • Mean: 0.84
  • Median: 0.88
  • Poor shutdown (>0.8): 1052

Using 10 features:
  avg_consumption, baseload, avg_peak_ratio, night_ratio, consumption_cv, avg_volatility, temp_sensitivity, building_type_encoded, heating_avg, cooling_avg



/databricks/python/lib/python3.10/site-packages/mlflow/models/signature.py:212: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  inputs = _infer_schema(model_input) if model_input is not None else None


Uploading artifacts:   0%|          | 0/5 [00:00<?, ?it/s]

✅ MAE: 0.035
✅ R²: 0.929
✅ Poor shutdown: 40 buildings (2.6%)

✅ ALL 3 ENHANCED MODELS TRAINED & LOGGED TO MLFLOW

Run ID for registration: f1edc376ef6b4468b881c8258c07cc84


In [0]:
# CELL 5: Register Model (SAME)
model_name = "building-weekend-efficiency-predictor"
model_uri = f"runs:/{production_run_id}/xgboost_model"

print(f"Registering model from run: {production_run_id}")

result = mlflow.register_model(model_uri, model_name)

print(f"✅ Model registered successfully!")
print(f"Model name: {model_name}")
print(f"Version: {result.version}")

Registering model from run: f1edc376ef6b4468b881c8258c07cc84


Successfully registered model 'dnv_energy_demo.default.building-weekend-efficiency-predictor'.


Uploading artifacts:   0%|          | 0/5 [00:00<?, ?it/s]

✅ Model registered successfully!
Model name: building-weekend-efficiency-predictor
Version: 1


Created version '1' of model 'dnv_energy_demo.default.building-weekend-efficiency-predictor'.


In [0]:
# CELL 6: Business Insights (ENHANCED)
print("="*70)
print("ENHANCED BUSINESS INSIGHTS")
print("="*70)

print(f"\n📊 DATASET:")
print(f"  • Buildings: {len(df_bldg)}")
print(f"  • Features: {df_bldg.shape[1]}")
if 'heating_avg' in df_bldg.columns:
    print(f"  • Heating meters: {df_bldg['heating_avg'].notna().sum()}")
if 'cooling_avg' in df_bldg.columns:
    print(f"  • Cooling meters: {df_bldg['cooling_avg'].notna().sum()}")

print(f"\n🎯 MODEL 1 - CLUSTERING:")
print(f"  • Clusters: {optimal_k}")
print(f"  • Silhouette: {sil_score:.3f}")
print(f"  • Features: {len(cluster_features)} (behavioral + operational)")

for i in range(optimal_k):
    cluster_data = df_bldg[df_bldg['cluster'] == i]
    print(f"\n  Cluster {i} ({len(cluster_data)} buildings):")
    print(f"    • Avg baseload: {cluster_data['baseload'].mean():.1f} kWh")
    print(f"    • Avg weekend ratio: {cluster_data['weekend_ratio'].mean():.2f}")
    print(f"    • Avg night ratio: {cluster_data['night_ratio'].mean():.2f}")

print(f"\n🎯 MODEL 2 - ANOMALY DETECTION:")
print(f"  • Features: {len(anomaly_features)}")
print(f"  • Anomalies: {n_anomalies} ({n_anomalies/len(df_bldg)*100:.1f}%)")

anomalies = df_bldg[df_bldg['is_anomaly'] == 1]
if len(anomalies) > 0:
    print(f"  • Avg baseload (anomalies): {anomalies['baseload'].mean():.1f} kWh")
    print(f"  • Avg baseload (normal): {df_bldg[df_bldg['is_anomaly']==0]['baseload'].mean():.1f} kWh")

print(f"\n🎯 MODEL 3 - WEEKEND EFFICIENCY:")
print(f"  • Features: {len(xgb_features)}")
print(f"  • MAE: {mae:.3f}, R²: {r2:.3f}")
print(f"  • Poor shutdown: {n_poor} buildings")

print(f"\n💰 TOP 10 UNDERPERFORMERS (weekend gap):")
if 'weekend_gap' in df_bldg.columns:
    top_10 = df_bldg.nlargest(10, 'weekend_gap')[
        ['building_id', 'building_type', 'weekend_ratio', 'predicted_weekend_ratio', 
         'weekend_gap', 'night_ratio', 'is_anomaly']
    ]
    print(top_10.to_string(index=False))

print("\n✅ ENHANCED PIPELINE COMPLETE")

ENHANCED BUSINESS INSIGHTS

📊 DATASET:
  • Buildings: 1533
  • Features: 28
  • Heating meters: 174
  • Cooling meters: 517

🎯 MODEL 1 - CLUSTERING:
  • Clusters: 3
  • Silhouette: 0.577
  • Features: 6 (behavioral + operational)

  Cluster 0 (140 buildings):
    • Avg baseload: 6.9 kWh
    • Avg weekend ratio: 0.54
    • Avg night ratio: 0.51

  Cluster 1 (87 buildings):
    • Avg baseload: 756.8 kWh
    • Avg weekend ratio: 0.91
    • Avg night ratio: 0.89

  Cluster 2 (1306 buildings):
    • Avg baseload: 76.2 kWh
    • Avg weekend ratio: 0.86
    • Avg night ratio: 0.82

🎯 MODEL 2 - ANOMALY DETECTION:
  • Features: 11
  • Anomalies: 230 (15.0%)
  • Avg baseload (anomalies): 251.6 kWh
  • Avg baseload (normal): 83.3 kWh

🎯 MODEL 3 - WEEKEND EFFICIENCY:
  • Features: 10
  • MAE: 0.035, R²: 0.929
  • Poor shutdown: 40 buildings

💰 TOP 10 UNDERPERFORMERS (weekend gap):
               building_id building_type  weekend_ratio  predicted_weekend_ratio  weekend_gap  night_ratio  is_anomaly